In [ ]:
import json
import pickle
import molecular_structure_clustering
import tmap as tm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from map4 import MAP4Calculator
from rdkit.Chem.PandasTools import LoadSDF

# 0. Load required molecules

In [ ]:
with open('data/' + 'original_molecules_Feb23.pkl', 'rb') as f:
    original_molecules = pickle.load(f)

In [ ]:
sdf_filename = 'data/SECRETed_space_Feb23.sdf'
df_complete = LoadSDF(sdf_filename, smilesName='SMILES', molColName='Molecule', strictParsing=True)
df_smiles = df_complete[['ID','SMILES', 'Molecule']]

# 1. Load distance matrix

In [ ]:
similarity_matrix = pd.read_csv('similarity_matrixv4.csv', index_col=0, header=0)

## 1.1. Optional: transform the similarity matrix to a distance matrix

In [ ]:
distance_matrix = similarity_matrix.applymap(lambda x: 1-x)

# 2. Calculate structure clustering

In [ ]:
complete_model = molecular_structure_clustering.structure_clustering(1, 'complete', distance_matrix = distance_matrix)

## 2.1. Get dendogram distances for later

In [ ]:
dendrogram_distances = molecular_structure_clustering.get_distances(complete_model)

## 2.2. Optional: plot dendogram

In [ ]:
fig = plt.figure(figsize=(30,200))
ax = fig.add_subplot()
# molecular_structure_clustering.plot_dendrogram(complete_model, ax=ax, orientation='left')

# 3. Get leaves

In [ ]:
children = complete_model.children_
leaves, children_new = molecular_structure_clustering.get_leaves(children)

## 3.1. Optional: store leaves

In [ ]:
# with open('backups/' + 'leaves_graphs.pkl', 'wb') as f:
#     pickle.dump(leaves, f)

In [ ]:
# with open('backups/' + 'leaves_graphs.pkl', 'rb') as f:
#     leaves = pickle.load(f)

# 4. Initialization

In [ ]:
substructures_smarts, substructures_molecules, substructures_cancelations = molecular_structure_clustering.get_substructure_list(df_smiles, leaves, rings=False)

## 4.1. Optional: store substructure smarts

In [ ]:
# with open('data/' + 'substructures_smarts_Apr23.pkl', 'wb') as f:
#     pickle.dump(substructures_smarts, f)

In [ ]:
# with open('data/' + 'substructures_smarts_Apr23.pkl', 'rb') as f:
#     substructures_smarts = pickle.load(f)

# 5. Get brothers

In [ ]:
brothers = molecular_structure_clustering.get_brothers(df_smiles, leaves, distance_matrix, dendrogram_distances, linkage='complete')

## 5.1. Optional: store brothers

In [ ]:
# with open('data/' + 'brothers_Apr23.json', 'w') as outfile:
#     json.dump(brothers, outfile)

In [ ]:
# with open('data/' + 'brothers_Apr23.json', 'r') as outfile:
#     brothers = json.load(outfile)

# 6. Apply MAP4

In [ ]:
dim = 1024
MAP4 = MAP4Calculator(dimensions=dim)
ENC = tm.Minhash(dim)

n_samples = df_smiles.shape[0]

# Computes the Fingerprints of the substructures
fps_substructures = MAP4.calculate_many(substructures_molecules)
fps_molecules = MAP4.calculate_many(original_molecules)

In [ ]:
results = []
for j in range(len(original_molecules)-1):
    print(f'Evaluating model: [{j}/{len(original_molecules)-1}]', end='\r')
    # fps_molecules WILL BE ORIGINAL GRAPHS AND fps_substructures WILL BE INTERSECTION GRAPHS
    results_j = molecular_structure_clustering.mcs_silhouette(j, fps_molecules, fps_substructures, leaves, brothers, ENC)
    results.append(results_j)

## 6.1. Optional: store MAP4 results

In [ ]:
# with open('data/' + 'silhouette_results_Apr23.pkl', 'wb') as f:
#     pickle.dump(results, f)

In [ ]:
# with open('data/' + 'silhouette_results_Apr23.pkl', 'rb') as f:
#     results = pickle.load(f)

## 6.2. Plot results

In [ ]:
fig, axis = plt.subplots(figsize=(20, 10))
X = np.abs(np.array([v['mean_silhouette'] for v in results]))
X_std = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
X_scaled = X_std * (1 - 0) + 0

axis.plot([v['n_clusters'] for v in results], X_scaled, label='mol2MCS dissimilarity', color='blue')

axis.plot([v['n_clusters'] for v in results], [v['mean_a'] for v in results], label='a mol2MCS', color='red')
axis.plot([v['n_clusters'] for v in results], [v['mean_b'] for v in results], label='b mol2MCS', color='black')
axis.legend()
axis.set_xlabel('Number of clusters')
axis.set_ylabel('Scaled Modified Silhouette Coefficient')

In [ ]:
ig, axis = plt.subplots(figsize=(20, 10))
X = np.abs(np.array([v['mean_silhouette'] for v in results]))
X_std = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
X_scaled = X_std * (1 - 0) + 0

# Compute the derivative of the silhouette scores
gradient = np.gradient(X_scaled)  # Find the index of the maximum value of the silhouette scores
max_index = np.argmax(X_scaled)   # Find the index of the first zero-crossing of the derivative
#ip = np.where(np.abs(gradient) < 0.00001)[0]
zeros = np.where(np.diff(np.sign(gradient)))[0]
max_derivative_index = zeros[0] # Plot the silhouette scores, derivative, and maximum point

axis.plot([v['n_clusters'] for v in results], X_scaled, label='mol2MCS dissimilarity', color='blue')
axis.plot([v['n_clusters'] for v in results], gradient, label='Derivative of Silhouette Scores')
axis.plot([v['n_clusters'] for v in results][max_index], X_scaled[max_index],'o',color='red')
axis.legend()
axis.set_xlabel('Number of clusters')
axis.set_ylabel('Scaled Modified Silhouette Coefficient')